In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([ 
    #function used to preprocess and transform images before sending them to nn
    # Apply these transformations one after another.
    
    transforms.ToTensor(),    
    #auto creates pytorch tensors and scale them (0,1)
    #It also changes the image representation to PyTorch's typical CHW format:Channels × Height × Width
    
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)) 
    #normalizes data -> mean = (0.5, 0.5, 0.5) & std  = (0.5, 0.5, 0.5)
])

trainset = CIFAR10(root='./Data', train=True, download=True, transform=transform) 
testset = CIFAR10(root='./Data', train=False, download=True, transform=transform)

#root -> This tells PyTorch where to store the dataset.
# transform=transform -> This applies the transformation you defined earlier:

In [3]:
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset, batch_size=64)

In [4]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./Data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

# Build CNN Model

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), #3  → input channels
                                                        #32 → output channels / filters
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel size=2, stride=2 -> it performs downsampling

            nn.Conv2d(32, 64, kernel_size=3, padding=1), #allow it to learn more feature maps
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel size=2, stride=2

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) #Increase channels while reducing spatial dimensions.
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256), #2048 features and converts them into 256 neurons.
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) #flattening # x.size(0) to get the batch size = 64
        x = self.fc_layers(x)

        return x

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training CNN

In [8]:
epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images) #FP
        loss = criterion(output, labels) #loss fnx
        loss.backward() #BP
        optimizer.step() #update params

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss={epoch_training_loss/len(train_loader)}")


epoch = 1/10 & loss=1.3582667502600823
epoch = 2/10 & loss=0.9337369753500385
epoch = 3/10 & loss=0.74879424792269
epoch = 4/10 & loss=0.621351213711302
epoch = 5/10 & loss=0.5131242854897019
epoch = 6/10 & loss=0.41693647274428314
epoch = 7/10 & loss=0.32891790483080213
epoch = 8/10 & loss=0.25372927392478034
epoch = 9/10 & loss=0.19327144651099695
epoch = 10/10 & loss=0.1531818694413623


## Evaluation

In [9]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad(): #no BP
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0) # number of samples in the current batch.

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 74.63
